In [ ]:
# Demo (PyTorch)

# Overview
- MNIST dataset
  + Goal: estimating summation of handwritten digit images
  + Input: a sequence of up to 8 images of size `28x28` (PyTorch format: `(N, 8, 1, 28, 28)`)
  + Output: summation of given handwritten digit images (regression)
- Defining a CNN+LSTM for MNIST dataset in PyTorch
- Training and evaluating a CNN+LSTM

# Import libraries

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# (1) Data

In [ ]:
# Load MNIST via torchvision
train_raw = datasets.MNIST(root='./data', train=True, download=True)
test_raw = datasets.MNIST(root='./data', train=False, download=True)

X_train_raw = train_raw.data.numpy().astype('float32') / 255.0
y_train_temp = train_raw.targets.numpy()

X_test_raw = test_raw.data.numpy().astype('float32') / 255.0
y_test_temp = test_raw.targets.numpy()

print('X_train_raw shape:', X_train_raw.shape)
print(X_train_raw.shape[0], 'train samples')
print(X_test_raw.shape[0], 'test samples')

## Original data
- Input: 28x28 image
- Output: label (0, 1,..., 9)

## Data preprocessing
- Read images from MNIST
- Sample from 1 to 8 images per sequence and compute their summation

## New data
- Input: sequence of 8 images `(8, 1, 28, 28)` (zero-padded if fewer than 8 digits)
- Output: summation of the input digit images

In [ ]:
n_train = X_train_raw.shape[0]
n_test = X_test_raw.shape[0]

np.random.seed(2018)
maxToAdd = 8
size = 28

# Dùng float32 để tiết kiệm 50% RAM so với float64; định dạng PyTorch: (N, T, C, H, W)
x_train = np.zeros((n_train, maxToAdd, 1, size, size), dtype=np.float32)
y_train = np.zeros((n_train, 1), dtype=np.float32)

for i in range(n_train):
    numToAdd = int(np.ceil(np.random.rand() * maxToAdd))
    indices = np.random.choice(X_train_raw.shape[0], size=numToAdd)
    x_train[i, 0:numToAdd, 0, :, :] = X_train_raw[indices]
    y_train[i, 0] = np.sum(y_train_temp[indices])

print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)

x_test = np.zeros((n_test, maxToAdd, 1, size, size), dtype=np.float32)
y_test = np.zeros((n_test, 1), dtype=np.float32)

for i in range(n_test):
    numToAdd = int(np.ceil(np.random.rand() * maxToAdd))
    indices = np.random.choice(X_test_raw.shape[0], size=numToAdd)
    x_test[i, 0:numToAdd, 0, :, :] = X_test_raw[indices]
    y_test[i, 0] = np.sum(y_test_temp[indices])

print('x_test shape :', x_test.shape)
print('y_test shape :', y_test.shape)

# (2) Declare model

In [ ]:
class MNIST_CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        # CNN feature extractor (áp dụng theo từng bước thời gian tương đương TimeDistributed)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv_dropout1 = nn.Dropout(0.2)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv_dropout2 = nn.Dropout(0.2)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv_dropout3 = nn.Dropout(0.2)

        # 256 * 3 * 3 = 2304
        self.lstm1 = nn.LSTM(input_size=256 * 3 * 3, hidden_size=128, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=128, batch_first=True)
        self.lstm_dropout = nn.Dropout(0.22)

        self.fc_out = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)

        x = self.conv_dropout1(self.pool1(F.relu(self.conv1(x))))
        x = self.conv_dropout2(self.pool2(F.relu(self.conv2(x))))
        x = self.conv_dropout3(self.pool3(F.relu(self.conv3(x))))

        x = x.flatten(start_dim=1).view(B, T, -1)  # (B, 8, 2304)

        x, _ = self.lstm1(x)       # (B, 8, 128) - return_sequences=True
        x, _ = self.lstm2(x)       # (B, 8, 128)
        x = x[:, -1, :]            # (B, 128)    - return_sequences=False (lấy bước cuối)
        x = self.lstm_dropout(x)
        return self.fc_out(x)      # (B, 1)

model = MNIST_CNN_LSTM().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params:,}')
# Ghi chú: PyTorch nn.LSTM dùng 2 vector bias (bias_ih & bias_hh) cho CuDNN nên nhiều hơn Keras đúng 1,024 params (1,748,097 vs 1,747,073), tương đương về mặt toán học.

In [ ]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, maxToAdd, 1, 28, 28), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

# (3) Train defined model
- Training history (`loss`, `val_loss`) được lưu trong biến `history`
- Nếu hệ thống thiếu VRAM (OOM), hãy giảm `batch_size` (ví dụ xuống `64` hoặc `32`)

In [ ]:
def evaluate_mae(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            preds = model(bx)
            loss = criterion(preds, by)
            total_loss += loss.item() * bx.size(0)
            total += bx.size(0)
    return total_loss / total


lr = 0.01
criterion = nn.L1Loss()  # mean_absolute_error
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

batch_size = 128
epochs = 30

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'val_loss': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, total = 0.0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        total += bx.size(0)

    train_loss = running_loss / total
    val_loss = evaluate_mae(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch:2d}/{epochs} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}')

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

# (4) Evaluate trained model

In [ ]:
test_mae = evaluate_mae(model, test_loader, criterion, device)
print('Mean absolute error of testing set:', test_mae)

In [ ]:
# select a test sequence randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]

plt.figure(figsize=(10, 3))
for indx in range(maxToAdd):
    plt.subplot(1, maxToAdd, indx + 1)
    test_img = x_test[random_test_index, indx, 0, :, :]
    plt.axis('off')
    plt.imshow(test_img, cmap='gray')

# estimate summation of testing images
model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(x_test[random_test_index], axis=0)).to(device)
    est_sum = model(inp).item()

print('> actual summation = %d' % int(y_test[random_test_index, 0]))
print('> estimated summation = %.3f' % est_sum)
plt.show()